<a href="https://colab.research.google.com/github/alejox1888/Proyecto-Final-IA/blob/main/notebooks/01_corpus_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Preparación del Corpus de Wikipedia en Español

**Proyecto Final - Inteligencia Artificial**
*Exploración de Embeddings y Similitud Semántica*

Este notebook descarga un subconjunto de la Wikipedia en español, lo limpia, lo tokeniza, construye el vocabulario aplicando subsampling de palabras frecuentes (Mikolov et al., 2013, sec. 2.3), y guarda los archivos procesados para los notebooks siguientes.

## Salidas que genera
- `data/corpus_tokens.npy`: corpus tokenizado como secuencia de IDs (int32).
- `data/vocab.json`: diccionario `word -> id` y `id -> word`.
- `data/word_freqs.npy`: frecuencias absolutas de cada palabra del vocabulario.
- `data/raw_articles.txt`: artículos crudos (uno por línea) para reutilización.




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/proyecto_ia_final"
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.chdir(PROJECT_DIR)

print(f"Trabajando en: {os.getcwd()}")

Mounted at /content/drive
Trabajando en: /content/drive/MyDrive/proyecto_ia_final


## 1. Instalación de dependencias

In [ ]:
!pip install datasets nltk -q

In [ ]:
import os
import re
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import nltk
from datasets import load_dataset
from tqdm.auto import tqdm

nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


## 2. Configuración

In [ ]:
# Hiperparámetros del corpus
N_ARTICLES = 50_000           # Número de artículos de Wikipedia a usar
MIN_WORD_LENGTH = 2            # Longitud mínima de palabra a conservar
MIN_COUNT = 10                 # Frecuencia mínima para entrar al vocabulario
SUBSAMPLING_THRESHOLD = 1e-5   # Umbral de subsampling (Mikolov, sec. 2.3)
MAX_VOCAB_SIZE = 80_000        # Límite superior del vocabulario

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Configuración:")
print(f"  Artículos a procesar:   {N_ARTICLES:,}")
print(f"  Frecuencia mínima:      {MIN_COUNT}")
print(f"  Umbral de subsampling:  {SUBSAMPLING_THRESHOLD}")
print(f"  Tamaño máx. vocab:      {MAX_VOCAB_SIZE:,}")


Configuración:
  Artículos a procesar:   50,000
  Frecuencia mínima:      10
  Umbral de subsampling:  1e-05
  Tamaño máx. vocab:      80,000


## 3. Descarga de Wikipedia en Español

Usamos el dataset `wikimedia/wikipedia` de HuggingFace con la versión española (`20231101.es`). Para no descargar los 5+ GB completos, usamos modo *streaming* y tomamos solo los primeros `N_ARTICLES` artículos.

In [ ]:
raw_path = DATA_DIR / "raw_articles.txt"

if raw_path.exists():
    print(f"Archivo {raw_path} ya existe. Saltando descarga.")
    with open(raw_path, "r", encoding="utf-8") as f:
        articles = [line.strip() for line in f if line.strip()]
    print(f"Cargados {len(articles):,} artículos desde disco.")
else:
    print("Descargando Wikipedia en español (streaming)...")
    ds = load_dataset("wikimedia/wikipedia", "20231101.es", split="train", streaming=True)

    articles = []
    for i, article in enumerate(tqdm(ds, total=N_ARTICLES, desc="Descargando")):
        if i >= N_ARTICLES:
            break
        # Reemplazar saltos de línea internos por espacios para que cada artículo quepa en una línea
        text = article['text'].replace('\n', ' ').replace('\r', ' ')
        articles.append(text)

    print(f"Guardando {len(articles):,} artículos en {raw_path}...")
    with open(raw_path, "w", encoding="utf-8") as f:
        for text in articles:
            f.write(text + "\n")
    print("Listo.")


Descargando Wikipedia en español (streaming)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Descargando:   0%|          | 0/50000 [00:00<?, ?it/s]

Guardando 50,000 artículos en data/raw_articles.txt...
Listo.


## 4. Limpieza y Tokenización

Aplicamos los siguientes pasos a cada artículo:
1. Pasar todo a minúsculas.
2. Eliminar números (los reemplazamos con un token `<NUM>` opcional).
3. Reemplazar caracteres no alfabéticos por espacios (mantenemos letras españolas: á, é, í, ó, ú, ñ, ü).
4. Tokenizar usando NLTK.
5. Filtrar palabras muy cortas.

In [ ]:
# Regex para limpieza
# Mantiene letras (incluyendo acentos y ñ) y espacios. Todo lo demás se reemplaza.
CLEAN_RE = re.compile(r"[^a-záéíóúñü\s]", re.IGNORECASE)

def clean_and_tokenize(text):
    """Limpia un texto y devuelve una lista de tokens."""
    text = text.lower()
    text = CLEAN_RE.sub(" ", text)
    tokens = text.split()  # split() es mucho más rápido que nltk para texto ya limpio
    tokens = [t for t in tokens if len(t) >= MIN_WORD_LENGTH]
    return tokens

print("Tokenizando artículos...")
all_tokens = []
for article in tqdm(articles, desc="Tokenizando"):
    all_tokens.extend(clean_and_tokenize(article))

print(f"\nTotal de tokens (antes de filtrar): {len(all_tokens):,}")


Tokenizando artículos...


Tokenizando:   0%|          | 0/50000 [00:00<?, ?it/s]


Total de tokens (antes de filtrar): 82,619,407


## 5. Construcción del Vocabulario

Contamos frecuencias, conservamos las palabras que aparecen al menos `MIN_COUNT` veces, y limitamos al tamaño máximo del vocabulario.

In [ ]:
print("Contando frecuencias...")
counter = Counter(all_tokens)
print(f"Palabras únicas (antes de filtrar): {len(counter):,}")

# Filtrar por frecuencia mínima
filtered_counts = [(w, c) for w, c in counter.most_common() if c >= MIN_COUNT]
print(f"Palabras con frecuencia >= {MIN_COUNT}: {len(filtered_counts):,}")

# Limitar tamaño del vocabulario
if len(filtered_counts) > MAX_VOCAB_SIZE:
    filtered_counts = filtered_counts[:MAX_VOCAB_SIZE]
    print(f"Vocabulario truncado a {MAX_VOCAB_SIZE:,} palabras.")

# Construir mapeos
word2id = {w: i for i, (w, _) in enumerate(filtered_counts)}
id2word = {i: w for w, i in word2id.items()}
word_freqs = np.array([c for _, c in filtered_counts], dtype=np.int64)

VOCAB_SIZE = len(word2id)
print(f"\nTamaño final del vocabulario: {VOCAB_SIZE:,}")
print(f"Palabras más frecuentes: {[w for w, _ in filtered_counts[:10]]}")


Contando frecuencias...
Palabras únicas (antes de filtrar): 843,805
Palabras con frecuencia >= 10: 157,533
Vocabulario truncado a 80,000 palabras.

Tamaño final del vocabulario: 80,000
Palabras más frecuentes: ['de', 'la', 'en', 'el', 'que', 'los', 'del', 'se', 'las', 'por']


## 6. Subsampling de Palabras Frecuentes

Mikolov et al. (2013) proponen descartar palabras frecuentes con probabilidad:

$$P(\text{descartar } w_i) = 1 - \sqrt{\frac{t}{f(w_i)}}$$

donde $t$ es el umbral (típicamente $10^{-5}$) y $f(w_i)$ es la frecuencia relativa de la palabra. Esto mejora la calidad de los embeddings al reducir la influencia desproporcionada de palabras como "el", "de", "la".

In [ ]:
total_count = word_freqs.sum()
word_probs = word_freqs / total_count  # frecuencia relativa

# Probabilidad de CONSERVAR cada palabra
keep_prob = np.sqrt(SUBSAMPLING_THRESHOLD / word_probs)
keep_prob = np.clip(keep_prob, 0, 1)

print("Probabilidades de conservación para las palabras más frecuentes:")
for i in range(10):
    w = id2word[i]
    print(f"  {w:15s} freq={word_freqs[i]:>8,}  P(keep)={keep_prob[i]:.4f}")


Probabilidades de conservación para las palabras más frecuentes:
  de              freq=6,943,985  P(keep)=0.0107
  la              freq=3,758,773  P(keep)=0.0146
  en              freq=2,915,394  P(keep)=0.0165
  el              freq=2,805,649  P(keep)=0.0169
  que             freq=1,587,433  P(keep)=0.0224
  los             freq=1,389,422  P(keep)=0.0240
  del             freq=1,284,444  P(keep)=0.0249
  se              freq=1,110,553  P(keep)=0.0268
  las             freq= 885,131  P(keep)=0.0300
  por             freq= 878,935  P(keep)=0.0301


In [ ]:
# Convertir el corpus a IDs, aplicando subsampling
print("Convirtiendo corpus a IDs y aplicando subsampling...")

corpus_ids = []
n_oov = 0  # out-of-vocabulary
n_subsampled = 0

for token in tqdm(all_tokens, desc="Procesando"):
    if token not in word2id:
        n_oov += 1
        continue
    idx = word2id[token]
    # Mantener con probabilidad keep_prob[idx]
    if random.random() < keep_prob[idx]:
        corpus_ids.append(idx)
    else:
        n_subsampled += 1

corpus_ids = np.array(corpus_ids, dtype=np.int32)

print(f"\nTokens originales:           {len(all_tokens):,}")
print(f"Tokens fuera de vocabulario: {n_oov:,}")
print(f"Tokens descartados (subsamp): {n_subsampled:,}")
print(f"Tokens finales en el corpus:  {len(corpus_ids):,}")


Convirtiendo corpus a IDs y aplicando subsampling...


Procesando:   0%|          | 0/82619407 [00:00<?, ?it/s]


Tokens originales:           82,619,407
Tokens fuera de vocabulario: 2,810,988
Tokens descartados (subsamp): 54,526,430
Tokens finales en el corpus:  25,281,989


## 7. Guardar Archivos Procesados

In [ ]:
# Guardar corpus
np.save(DATA_DIR / "corpus_tokens.npy", corpus_ids)
print(f"Corpus guardado en {DATA_DIR / 'corpus_tokens.npy'} ({corpus_ids.nbytes / 1e6:.1f} MB)")

# Guardar vocabulario
vocab_data = {
    "word2id": word2id,
    "id2word": {str(k): v for k, v in id2word.items()},  # JSON requiere strings como keys
    "vocab_size": VOCAB_SIZE,
}
with open(DATA_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_data, f, ensure_ascii=False, indent=2)
print(f"Vocabulario guardado en {DATA_DIR / 'vocab.json'}")

# Guardar frecuencias
np.save(DATA_DIR / "word_freqs.npy", word_freqs)
print(f"Frecuencias guardadas en {DATA_DIR / 'word_freqs.npy'}")

print("\n¡Notebook 1 completado!")


Corpus guardado en data/corpus_tokens.npy (101.1 MB)
Vocabulario guardado en data/vocab.json
Frecuencias guardadas en data/word_freqs.npy

¡Notebook 1 completado!


## 8. Validación: Estadísticas del Corpus

Imprimimos algunas estadísticas para verificar que todo quedó bien.

In [ ]:
print("=" * 60)
print("RESUMEN DEL CORPUS PROCESADO")
print("=" * 60)
print(f"Tamaño del vocabulario:      {VOCAB_SIZE:,}")
print(f"Tokens en el corpus final:   {len(corpus_ids):,}")
print(f"Palabra más frecuente:       '{id2word[0]}' ({word_freqs[0]:,} veces)")
print(f"Palabra menos frecuente:     '{id2word[VOCAB_SIZE-1]}' ({word_freqs[-1]:,} veces)")
print(f"\nPrimeros 30 tokens del corpus (como palabras):")
print(" ".join(id2word[i] for i in corpus_ids[:30]))


RESUMEN DEL CORPUS PROCESADO
Tamaño del vocabulario:      80,000
Tokens en el corpus final:   25,281,989
Palabra más frecuente:       'de' (6,943,985 veces)
Palabra menos frecuente:     'apologético' (31 veces)

Primeros 30 tokens del corpus (como palabras):
andorra oficialmente principado andorra micro soberano litoral ubicado españa límite península ibérica parlamentario organizado parroquias habitantes febrero andorra sus extensión andorra micro pirineos francia tiene ms limita españa catalana lérida
